# Visual 2: Living Area vs. Price Relationship

This scatter plot explores how property size and amenities influence housing prices, showing that larger homes with better features and proximity to strong school districts tend to command higher prices.

In [ ]:
%pip install scipy

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
from scipy import stats

# Enable tooltips
alt.data_transformers.enable('default')
alt.renderers.enable('default')

In [ ]:
# Load the housing data
df = pd.read_csv('../data/austin_housing_cleaned.csv')

# Display initial data info
print(f"Total records: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Prepare data for visualization
# Filter out records with missing or invalid values
viz_data = df[
    (df['livingAreaSqFt'].notna()) & 
    (df['latestPrice'].notna()) & 
    (df['livingAreaSqFt'] > 0) & 
    (df['latestPrice'] > 0) &
    (df['avgSchoolRating'].notna())
].copy()

# Remove outliers using IQR method for both livingAreaSqFt and latestPrice to prevent graph stretching
print(f"Before outlier removal: {len(viz_data)} records")
print(f"Living Area Range: {viz_data['livingAreaSqFt'].min():.0f} - {viz_data['livingAreaSqFt'].max():.0f} sq ft")
print(f"Price Range: ${viz_data['latestPrice'].min():,.0f} - ${viz_data['latestPrice'].max():,.0f}")

# Remove living area outliers
Q1_area = viz_data['livingAreaSqFt'].quantile(0.25)
Q3_area = viz_data['livingAreaSqFt'].quantile(0.75)
IQR_area = Q3_area - Q1_area
lower_area = Q1_area - 1.5 * IQR_area
upper_area = Q3_area + 1.5 * IQR_area

# Remove price outliers
Q1_price = viz_data['latestPrice'].quantile(0.25)
Q3_price = viz_data['latestPrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_price = Q1_price - 1.5 * IQR_price
upper_price = Q3_price + 1.5 * IQR_price

# Store outliers for reporting
outliers = viz_data[
    (viz_data['livingAreaSqFt'] < lower_area) | 
    (viz_data['livingAreaSqFt'] > upper_area) |
    (viz_data['latestPrice'] < lower_price) |
    (viz_data['latestPrice'] > upper_price)
].copy()

# Filter out extreme values (outliers)
viz_data = viz_data[
    (viz_data['livingAreaSqFt'] >= lower_area) & 
    (viz_data['livingAreaSqFt'] <= upper_area) &
    (viz_data['latestPrice'] >= lower_price) &
    (viz_data['latestPrice'] <= upper_price)
].copy()

print(f"\nAfter outlier removal: {len(viz_data)} records")
print(f"Living Area Range: {viz_data['livingAreaSqFt'].min():.0f} - {viz_data['livingAreaSqFt'].max():.0f} sq ft")
print(f"Price Range: ${viz_data['latestPrice'].min():,.0f} - ${viz_data['latestPrice'].max():,.0f}")
print(f"\nOutliers removed: {len(outliers)} properties")
if len(outliers) > 0:
    print(f"Outlier price range: ${outliers['latestPrice'].min():,.0f} - ${outliers['latestPrice'].max():,.0f}")
    print(f"Outlier area range: {outliers['livingAreaSqFt'].min():.0f} - {outliers['livingAreaSqFt'].max():.0f} sq ft")

# Create school quality categories for better visualization
viz_data['school_quality'] = pd.cut(
    viz_data['avgSchoolRating'],
    bins=[0, 3, 5, 7, 10],
    labels=['Below Average (0-3)', 'Average (3-5)', 'Good (5-7)', 'Excellent (7-10)'],
    include_lowest=True
)

# Create amenity score (sum of various amenity indicators)
amenity_cols = ['hasGarage', 'hasCooling', 'hasHeating', 'hasSpa', 'hasView']
viz_data['amenity_score'] = 0
for col in amenity_cols:
    if col in viz_data.columns:
        viz_data['amenity_score'] += viz_data[col].map({'TRUE': 1, 'FALSE': 0, True: 1, False: 0}).fillna(0)

print(f"Filtered records: {len(viz_data)}")
print(f"School quality distribution:\n{viz_data['school_quality'].value_counts().sort_index()}")

In [ ]:
# Calculate regression line and R²
x = viz_data['livingAreaSqFt'].values
y = viz_data['latestPrice'].values

# Calculate linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
r_squared = r_value ** 2

# Create regression line data
x_min, x_max = x.min(), x.max()
regression_df = pd.DataFrame({
    'livingAreaSqFt': [x_min, x_max],
    'predicted_price': [slope * x_min + intercept, slope * x_max + intercept]
})

print(f"R² value: {r_squared:.4f}")
print(f"Slope: ${slope:.2f} per sq ft")

In [ ]:
# viz_data.to_csv("../data/viz_data.csv", index=False)

# (Optional) save regression data for reuse
# regression_df.to_csv("../data/regression_data.csv", index=False)

In [ ]:
import altair as alt
import pandas as pd

# Create brush selection
brush = alt.selection_interval(name='brush')

# Optionally avoid Altair's default max rows limit for large datasets
try:
    alt.data_transformers.disable_max_rows()
except Exception:
    pass

# Define color palette for school quality (categorical palette for accessibility)
school_colors = {
    'Below Average (0-3)': '#e74c3c',  # Red
    'Average (3-5)': '#f39c12',       # Orange
    'Good (5-7)': '#3498db',          # Blue
    'Excellent (7-10)': '#2ecc71'     # Green
}

# Helpful for positioning and optional explicit x-domain
x_min = float(viz_data['livingAreaSqFt'].min())
x_max = float(viz_data['livingAreaSqFt'].max())

# Create scatter plot
scatter = alt.Chart(viz_data).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X(
        'livingAreaSqFt:Q',
        title='Living Area (sq ft)',
        scale=alt.Scale(domain=[x_min, x_max], zero=False)
    ),
    y=alt.Y(
        'latestPrice:Q',
        title='Property Price ($)',
        axis=alt.Axis(format='$,.0f'),
        scale=alt.Scale(zero=False)
    ),
    color=alt.condition(
        brush,
        alt.Color(
            'school_quality:N',
            title='School Quality Rating',
            scale=alt.Scale(domain=list(school_colors.keys()),
                            range=list(school_colors.values())),
            legend=alt.Legend(
                orient='bottom-right',
                title='School Quality',
                direction='vertical',
                titleFontSize=12,
                labelFontSize=11,
                padding=10,
                cornerRadius=5,
                fillColor='white',
                strokeColor='black'
            )
        ),
        alt.value('lightgray')
    ),
    tooltip=[
        alt.Tooltip('streetAddress:N', title='Address'),
        alt.Tooltip('city:N', title='City'),
        alt.Tooltip('livingAreaSqFt:Q', title='Living Area (sq ft)', format=','),
        alt.Tooltip('latestPrice:Q', title='Price', format='$,.0f'),
        alt.Tooltip('numOfBedrooms:Q', title='Bedrooms'),
        alt.Tooltip('numOfBathrooms:Q', title='Bathrooms'),
        alt.Tooltip('avgSchoolRating:Q', title='School Rating', format='.2f'),
        alt.Tooltip('amenity_score:Q', title='Amenity Score'),
        alt.Tooltip('yearBuilt:Q', title='Year Built')
    ]
).properties(
    width=700,
    height=450,
    title='Living Area vs. Price Relationship (Brush to select homes)'
)

# Create regression line
regression_line = alt.Chart(regression_df).mark_line(
    color='black',
    strokeDash=[5, 5],
    size=4
).encode(
    x='livingAreaSqFt:Q',
    y='predicted_price:Q'
)

# ---------- R² label slightly RIGHT of the first regression point (1.5% of range) ----------
# First point on the regression line (minimum X)
_min_x_idx = regression_df['livingAreaSqFt'].idxmin()
min_x = float(regression_df.loc[_min_x_idx, 'livingAreaSqFt'])
min_y = float(regression_df.loc[_min_x_idx, 'predicted_price'])

# Move label 2.5% of the x-range to the left of the first point
label_x = min_x - 0.075 * (x_max - x_min)
min_y = min_y - 8 * (x_max - x_min)

r2_text = alt.Chart(pd.DataFrame({
    'x': [label_x],
    'y': [min_y],
    'text': [f'R² = {r_squared:.4f}']
})).mark_text(
    align='left',        # render text to the right of the anchor
    baseline='middle',
    dx=2,                # tiny nudge right
    dy=-6,               # slight lift so it doesn't sit on the line
    fontSize=16,
    fontWeight='bold'          # plain black text (no bold, no stroke)
).encode(
    x='x:Q',
    y='y:Q',
    text='text:N'
)
# ---------------------------------------------------------------------------

# Combine scatter plot with regression line and annotation
scatter_chart = scatter + regression_line + r2_text

# Create linked histogram (filtered by brush)
histogram = alt.Chart(viz_data).mark_bar().encode(
    x=alt.X('latestPrice:Q',
            bin=alt.Bin(maxbins=30),
            title='Property Price ($)',
            axis=alt.Axis(format='$,.0f')),
    y=alt.Y('count():Q', title='Number of Homes'),
    color=alt.condition(
        brush,
        alt.value('#3498db'),
        alt.value('lightgray')
    ),
    tooltip=[
        alt.Tooltip('count()', title='Count'),
        alt.Tooltip('bin_maxbins_30_latestPrice:Q', title='Price (bin start)', format='$,.0f'),
        alt.Tooltip('bin_maxbins_30_latestPrice_end:Q', title='Price (bin end)', format='$,.0f')
    ]
).properties(
    width=700,
    height=150,
    title='Price Distribution (filtered by brush selection above)'
).transform_filter(brush)

# Combine both charts vertically and attach the selection at the top level
final_chart = alt.vconcat(
    scatter_chart,
    histogram
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_title(
    fontSize=15,
    anchor='start'
).configure_legend(
    titleFontSize=12,
    labelFontSize=11
).add_params(brush).properties(
    padding={"left": 20, "right": 10, "top": 10, "bottom": 10}
)

final_chart


#### Save Altair Chart to JSON

In [ ]:
import altair as alt
import os

os.makedirs("../reports/exports", exist_ok=True)
alt.data_transformers.enable("default", max_rows=None)

# Size each child (not the vconcat)
scatter_chart = scatter_chart.properties(width="container", height=450)
histogram     = histogram.properties(width="container", height=160)

# Rebuild the final chart
final_chart = alt.vconcat(scatter_chart, histogram, spacing=8).properties(
    autosize=alt.AutoSizeParams(type="fit", contains="padding", resize=True)
)

# Save as Vega-Lite JSON
final_chart.save("../reports/exports/LivingArea_vs_Price_Visual.Altair.json")
print("Saved to exports/LivingArea_vs_Price_Visual.Altair.json")


In [ ]:
%pip install "vegafusion[embed]>=1.5.0"
%pip install "vl-convert-python>=1.6.0"
